# 05 - Preprocessing Pipeline

## Objective

The purpose of this notebook is to prepare the engineered features
for machine learning.

- Separate input features and target variables
- Split the dataset into training and testing sets
- Identify numerical and categorical features
- Handle numerical features
- Encode categorical features
- Build a reusable preprocessing pipeline
- Transform the data
- Save the preprocessing pipeline for future predictions

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

import joblib

In [22]:
#Load feature-engineering dataset

X = pd.read_csv("../data/processed/final_features.csv")

y = pd.read_csv("../data/processed/targets.csv")

In [23]:
print("X shape:", X.shape)

print("y shape:", y.shape)

X shape: (97297, 36)
y shape: (97297, 5)


In [24]:
X.head()

,Unnamed: 0,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,...,glucose_postprandial,insulin_level,hba1c,pulse_pressure,bmi_category,mean_arterial_pressure,glucose_difference,total_to_hdl_ratio,activity_level,sleep_category
0,0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,...,236,6.36,8.18,56,Obese,96.666667,100,5.829268,Moderate,Optimal
1,1,52,Female,White,Highschool,Middle,Employed,Former,1,143,...,150,2.00,5.63,53,Normal,93.666667,57,2.109091,Low,Optimal
2,2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,...,195,5.07,7.51,42,Normal,87.000000,77,3.227273,Low,High
3,3,74,Female,Black,Highschool,Low,Retired,Never,0,49,...,253,5.28,9.03,27,Overweight,102.000000,114,3.420000,Low,Optimal
4,4,46,Male,White,Graduate,Middle,Retired,Never,1,109,...,184,12.74,7.20,25,Normal,75.333333,47,4.038462,Low,Optimal


In [25]:
y.head()

,diabetes_risk_score,hypertension_risk_score,heart_disease_risk_score,obesity_risk_score,cholesterol_risk_score
0,51.716583,53.581369,50.796020,57.387241,51.492983
1,26.096162,47.946740,23.853485,36.219202,3.984993
2,54.830087,42.141104,38.977694,42.898797,21.793300
3,60.729521,60.286946,53.579555,52.194082,22.962029
4,32.778085,15.279074,18.721942,32.095465,34.296907


In [26]:
X.dtypes

Unnamed: 0                              int64
Age                                     int64
gender                                    str
ethnicity                                 str
education_level                           str
income_level                              str
employment_status                         str
smoking_status                            str
alcohol_consumption_per_week            int64
physical_activity_minutes_per_week      int64
diet_score                            float64
sleep_hours_per_day                   float64
screen_time_hours_per_day             float64
family_history_diabetes                 int64
hypertension_history                    int64
cardiovascular_history                  int64
bmi                                   float64
waist_to_hip_ratio                    float64
systolic_bp                             int64
diastolic_bp                            int64
heart_rate                              int64
cholesterol_total                 

In [27]:
numerical_features = X.select_dtypes(
    include = ["int64", "float64"]
).columns.tolist()

In [28]:
categorical_features = X.select_dtypes(
    include = ["object", "category"]
).columns.tolist()

C:\Users\DELL\AppData\Local\Temp\ipykernel_7756\2262040447.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [29]:
print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Unnamed: 0', 'Age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c', 'pulse_pressure', 'mean_arterial_pressure', 'glucose_difference', 'total_to_hdl_ratio']

Categorical features:
['gender', 'ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status', 'bmi_category', 'activity_level', 'sleep_category']


In [14]:
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 27
Number of categorical features: 9


In [30]:
#Create the split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42
)

In [31]:
#Check the Split
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (77837, 36)
X_test : (19460, 36)
y_train: (77837, 5)
y_test : (19460, 5)


In [32]:
#Numerical Preprocessing 
numerical_transformer = StandardScaler()

In [35]:
#Categorical Preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [ ]:
#Combine the Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [39]:
#Create the Full Pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

In [40]:
# Fit preprocessing pipeline using training data only

preprocessing_pipeline.fit(X_train)

print("Preprocessing pipeline fitted successfully.")

Preprocessing pipeline fitted successfully.


In [41]:
#Transform Training Data

X_train_processed = preprocessing_pipeline.transform(X_train)

In [42]:
#Transform Testing Data

X_test_processed = preprocessing_pipeline.transform(X_test)

In [43]:
#Check the process data

print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)

print("Original X_test shape:", X_test.shape)
print("Processed X_test shape:", X_test_processed.shape)

Original X_train shape: (77837, 36)
Processed X_train shape: (77837, 61)
Original X_test shape: (19460, 36)
Processed X_test shape: (19460, 61)


In [44]:
#Convert Processed Data to DataFrames

feature_names = preprocessing_pipeline.get_feature_names_out()

print("Feature names after preprocessing:", feature_names)
print("Number of processed features:", len(feature_names))

Feature names after preprocessing: ['num__Unnamed: 0' 'num__Age' 'num__alcohol_consumption_per_week'
 'num__physical_activity_minutes_per_week' 'num__diet_score'
 'num__sleep_hours_per_day' 'num__screen_time_hours_per_day'
 'num__family_history_diabetes' 'num__hypertension_history'
 'num__cardiovascular_history' 'num__bmi' 'num__waist_to_hip_ratio'
 'num__systolic_bp' 'num__diastolic_bp' 'num__heart_rate'
 'num__cholesterol_total' 'num__hdl_cholesterol' 'num__ldl_cholesterol'
 'num__triglycerides' 'num__glucose_fasting' 'num__glucose_postprandial'
 'num__insulin_level' 'num__hba1c' 'num__pulse_pressure'
 'num__mean_arterial_pressure' 'num__glucose_difference'
 'num__total_to_hdl_ratio' 'cat__gender_Female' 'cat__gender_Male'
 'cat__gender_Other' 'cat__ethnicity_Asian' 'cat__ethnicity_Black'
 'cat__ethnicity_Hispanic' 'cat__ethnicity_Other' 'cat__ethnicity_White'
 'cat__education_level_Graduate' 'cat__education_level_Highschool'
 'cat__education_level_No formal' 'cat__education_level_Po

In [45]:
#Convert training data to DataFrame

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

In [46]:
#Convert test data to DataFrame

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [47]:
#processed dataf

print("Processed training data:")
X_train_processed_df.head()

Processed training data:


,num__Unnamed: 0,num__Age,num__alcohol_consumption_per_week,num__physical_activity_minutes_per_week,num__diet_score,num__sleep_hours_per_day,num__screen_time_hours_per_day,num__family_history_diabetes,num__hypertension_history,num__cardiovascular_history,...,cat__bmi_category_Normal,cat__bmi_category_Obese,cat__bmi_category_Overweight,cat__bmi_category_Underweight,cat__activity_level_High,cat__activity_level_Low,cat__activity_level_Moderate,cat__sleep_category_High,cat__sleep_category_Low,cat__sleep_category_Optimal
2647,-1.637462,0.632422,0.700310,-0.709157,0.563417,0.916872,-0.201211,-0.531957,-0.578473,-0.293364,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
32865,-0.562760,-1.367629,-0.709420,-0.673642,2.023393,-0.820604,-0.890148,-0.531957,-0.578473,-0.293364,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
24500,-0.860261,-2.012807,-0.709420,0.652223,0.956487,1.191210,0.528252,-0.531957,-0.578473,-0.293364,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2038,-1.659121,0.503387,1.405175,-0.010710,0.114193,-1.369281,-2.227497,-0.531957,-0.578473,-0.293364,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
45123,-0.126804,-0.399862,-1.414285,0.581195,0.619570,1.556995,0.163521,1.879852,1.728690,-0.293364,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [49]:
print("Processed training data:")
print(X_train_processed_df.shape)

print("\nProcessed testing data:")
print(X_test_processed_df.shape)

print(
    X_train_processed_df.shape[1]
    ==
    X_test_processed_df.shape[1]
)

Processed training data:
(77837, 61)

Processed testing data:
(19460, 61)
True


In [50]:
#Check for Missing Values and Infinites

print(
    "Missing values in training:",
    X_train_processed_df.isnull().sum().sum()
)

print(
    "Infinite values in training:",
    np.isinf(X_train_processed_df).sum().sum()
)

print(
    "Missing values in testing:",
    X_test_processed_df.isnull().sum().sum()
)

print(
    "Infinite values in testing:",
    np.isinf(X_test_processed_df).sum().sum()
)

Missing values in training: 0
Infinite values in training: 0
Missing values in testing: 0
Infinite values in testing: 0


In [51]:
#Check categorical encoding

encoded_columns = [
    column for column in feature_names
    if column.startswith("cat__")
]

print("Number of encoded categorical columns:", len(encoded_columns))

print("\nEncoded categorical columns:")
print(encoded_columns)

Number of encoded categorical columns: 34

Encoded categorical columns:
['cat__gender_Female', 'cat__gender_Male', 'cat__gender_Other', 'cat__ethnicity_Asian', 'cat__ethnicity_Black', 'cat__ethnicity_Hispanic', 'cat__ethnicity_Other', 'cat__ethnicity_White', 'cat__education_level_Graduate', 'cat__education_level_Highschool', 'cat__education_level_No formal', 'cat__education_level_Postgraduate', 'cat__income_level_High', 'cat__income_level_Low', 'cat__income_level_Lower-Middle', 'cat__income_level_Middle', 'cat__income_level_Upper-Middle', 'cat__employment_status_Employed', 'cat__employment_status_Retired', 'cat__employment_status_Student', 'cat__employment_status_Unemployed', 'cat__smoking_status_Current', 'cat__smoking_status_Former', 'cat__smoking_status_Never', 'cat__bmi_category_Normal', 'cat__bmi_category_Obese', 'cat__bmi_category_Overweight', 'cat__bmi_category_Underweight', 'cat__activity_level_High', 'cat__activity_level_Low', 'cat__activity_level_Moderate', 'cat__sleep_catego

In [52]:
#Check Target Variable 

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTarget columns:")
print(y_train.columns.tolist())

y_train shape: (77837, 5)
y_test shape: (19460, 5)

Target columns:
['diabetes_risk_score', 'hypertension_risk_score', 'heart_disease_risk_score', 'obesity_risk_score', 'cholesterol_risk_score']


In [54]:
#Check Target Data

print(y_train.head())

print("\n")

print(y_test.head())

       diabetes_risk_score  hypertension_risk_score  heart_disease_risk_score  \
2647             48.338757                47.768502                 37.549097   
32865            35.814028                40.867033                  7.106288   
24500            22.635144                25.396608                 22.432311   
2038             34.354236                30.864127                 21.353155   
45123            42.804727                37.438300                 23.498070   

       obesity_risk_score  cholesterol_risk_score  
2647            56.048058               25.509860  
32865           57.231693                4.389776  
24500           38.271332               20.064259  
2038            41.986217                3.967894  
45123           45.815670               26.553654  


       diabetes_risk_score  hypertension_risk_score  heart_disease_risk_score  \
11043            24.402726                34.279777                 32.207878   
4911             58.724192           

In [55]:
# Save train/test splits

X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Train/test datasets saved successfully.")

Train/test datasets saved successfully.


In [56]:
# Save preprocessing pipeline

joblib.dump(
    preprocessing_pipeline,
    "../models/preprocessing_pipeline.pkl"
)

print("Preprocessing pipeline saved successfully.")

Preprocessing pipeline saved successfully.


In [59]:
# Save feature names

joblib.dump(
    feature_names,
    "../models/feature_columns.pkl"
)

print("Feature names saved successfully.")

Feature names saved successfully.


In [60]:
# Load saved preprocessing pipeline

loaded_pipeline = joblib.load(
    "../models/preprocessing_pipeline.pkl"
)


# Verify loaded pipeline

X_test_verification = loaded_pipeline.transform(X_test)

if np.array_equal(
    X_test_processed,
    X_test_verification
):
    print("Pipeline verification successful.")
else:
    print("Pipeline verification failed.")

Pipeline verification successful.


In [61]:
#Final Verification

print("FINAL PREPROCESSING SUMMARY ")

print("Original dataset:")
print("X:", X.shape)
print("y:", y.shape)

print("\nTraining data:")
print("X_train:", X_train_processed_df.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test_processed_df.shape)
print("y_test:", y_test.shape)

print("\nNumber of processed features:")
print(len(feature_names))

FINAL PREPROCESSING SUMMARY 
Original dataset:
X: (97297, 36)
y: (97297, 5)

Training data:
X_train: (77837, 61)
y_train: (77837, 5)

Testing data:
X_test: (19460, 61)
y_test: (19460, 5)

Number of processed features:
61


## Data Leakage Prevention

The dataset was split into training and testing sets before fitting
the preprocessing pipeline.

The preprocessing pipeline was fitted only on the training data.

The same fitted pipeline was then used to transform the testing data.

Therefore, information from the testing dataset was not used when
learning preprocessing parameters.

## Final Notebook Summary

# Preprocessing Summary

## Input

The feature-engineered dataset contains the input features and the
five health risk score targets.

## Train-Test Split

The data was divided into:

- 80% training data
- 20% testing data

A fixed random state of 42 was used for reproducibility.

## Numerical Preprocessing

Numerical features were standardized using `StandardScaler`.

## Categorical Preprocessing

Categorical features were converted into numerical representations
using `OneHotEncoder`.

Unknown categories are ignored using:

`handle_unknown="ignore"`

## Pipeline

A Scikit-learn `ColumnTransformer` was used to apply different
transformations to numerical and categorical features.

The preprocessing steps were wrapped inside a reusable `Pipeline`.

## Data Leakage Prevention

The preprocessing pipeline was fitted only on the training dataset.
The testing dataset was transformed using the already-fitted pipeline.

## Saved Files

- `data/processed/X_train.csv`
- `data/processed/X_test.csv`
- `data/processed/y_train.csv`
- `data/processed/y_test.csv`
- `models/preprocessing_pipeline.pkl`
- `models/feature_columns.pkl`

## Next Step

The processed training data will be used in
`06_Model_Training.ipynb` to train and compare machine learning models.